**TASK:1**
Problem:
Before SupportAI can answer questions, it needs a structured FAQ knowledge base and a way to search it. Your first task is to create the data foundation and implement basic keyword-based search.

1. FAQ Knowledge Base
Create a FAQ knowledge base with at least 5 entries. Each entry must contain:

Field	Type	Description
id	string	Unique identifier (e.g. "faq-001")
category	string	Topic grouping (e.g. "Billing")
question	string	The FAQ question
answer	string	The official answer
keywords	list	Related search terms
Use the sample data provided in DESCRIPTION.md, or extend it with your own entries.

In [51]:

faqs = [
    {
        "id": "faq-001",
        "category": "Account",
        "question": "How do I reset my password?",
        "answer": (
            "Click 'Forgot Password' on the login page. "
            "Enter your registered email address and "
            "follow the password reset instructions."
        ),
        "keywords": [
            "password",
            "forgot",
            "reset",
            "login",
            "account"
        ]
    },

    {
        "id": "faq-002",
        "category": "Billing",
        "question": "What is your refund policy?",
        "answer": (
            "We offer full refunds within 30 days for "
            "unused subscriptions."
        ),
        "keywords": [
            "refund",
            "billing",
            "payment",
            "money",
            "subscription"
        ]
    },

    {
        "id": "faq-003",
        "category": "Delivery",
        "question": "How can I track my order?",
        "answer": (
            "Open the Orders page and click "
            "'Track Order' to view the latest status."
        ),
        "keywords": [
            "track",
            "delivery",
            "order",
            "shipping"
        ]
    },

    {
        "id": "faq-004",
        "category": "Technical",
        "question": "Why is the app crashing?",
        "answer": (
            "Update the application to the latest version "
            "and restart your device."
        ),
        "keywords": [
            "crash",
            "technical",
            "bug",
            "app",
            "error"
        ]
    },

    {
        "id": "faq-005",
        "category": "Account",
        "question": "How do I change my email address?",
        "answer": (
            "Go to Account Settings and choose "
            "'Update Email Address'."
        ),
        "keywords": [
            "email",
            "change",
            "account",
            "update"
        ]
    }
]

In [52]:
print(f"Total FAQs : {len(faqs)}")

for faq in faqs:
    print(f"{faq['id']} -> {faq['question']}")

Total FAQs : 5
faq-001 -> How do I reset my password?
faq-002 -> What is your refund policy?
faq-003 -> How can I track my order?
faq-004 -> Why is the app crashing?
faq-005 -> How do I change my email address?


In [53]:
def normalize_text(text):
    """Convert text to lowercase and remove extra spaces."""
    return text.strip().lower()

In [54]:
print(normalize_text(" Password "))
print(normalize_text("Refund"))
print(normalize_text(" ACCOUNT "))

password
refund
account


2. Search Functions
Implement the following functions:

search_by_keyword(faqs, query) -> list

Search FAQs by matching query words against keywords, question, and category fields.
Matching must be case-insensitive.
Return matching FAQs ordered by number of keyword hits (most matches first).
get_faq_by_id(faqs, faq_id) -> dict | None

Return the FAQ with the given id, or None if not found.
get_faqs_by_category(faqs, category) -> list

Return all FAQs belonging to the given category (case-insensitive)

In [55]:
def search_by_keyword(faqs, query):
    """
    Search FAQs by matching the query with keywords,
    question, and category (case-insensitive).

    Returns:
        list: Matching FAQs sorted by relevance.
    """
    query_words = normalize_text(query).split()

    matched_faqs = []

    for faq in faqs:
        searchable_text = " ".join([
            faq["category"],
            faq["question"],
            " ".join(faq["keywords"])
        ]).lower()

        score = 0

        for word in query_words:
            if word in searchable_text:
                score += 1

        if score > 0:
            matched_faqs.append((score, faq))

    matched_faqs.sort(key=lambda x: x[0], reverse=True)

    return [faq for score, faq in matched_faqs]

In [56]:
def get_faq_by_id(faqs, faq_id):
    """
    Return the FAQ with the given ID.
    Return None if the ID is not found.
    """
    faq_id = normalize_text(faq_id)

    for faq in faqs:
        if normalize_text(faq["id"]) == faq_id:
            return faq

    return None

In [57]:
def get_faqs_by_category(faqs, category):
    """
    Return all FAQs belonging to the given category.
    Matching is case-insensitive.
    """
    category = normalize_text(category)

    matched_faqs = []

    for faq in faqs:
        if normalize_text(faq["category"]) == category:
            matched_faqs.append(faq)

    return matched_faqs

3. Demonstration
Test your search functions with at least these queries:

Query	Expected Result
"forgot my password"	Password reset FAQ
"refund"	Refund policy FAQ
"weather today"	No match
Print the results clearly, showing the category, question, and answer for each match.

In [58]:
test_queries = [
    "forgot my password",
    "refund",
    "weather today"
]

for query in test_queries:
    print(f"\nQuery: {query}")

    results = search_by_keyword(faqs, query)

    if results:
        for faq in results:
            print(f"[{faq['category']}] {faq['question']}")
            print(f"→ {faq['answer']}")
    else:
        print("No matching FAQs found.")


Query: forgot my password
[Account] How do I reset my password?
→ Click 'Forgot Password' on the login page. Enter your registered email address and follow the password reset instructions.
[Delivery] How can I track my order?
→ Open the Orders page and click 'Track Order' to view the latest status.
[Account] How do I change my email address?
→ Go to Account Settings and choose 'Update Email Address'.

Query: refund
[Billing] What is your refund policy?
→ We offer full refunds within 30 days for unused subscriptions.

Query: weather today
No matching FAQs found.


**TASK:2**
Problem:
Keyword search can find the right FAQ, but returning raw FAQ text feels robotic. SupportAI needs to use a large language model to rephrase FAQ answers into natural, conversational responses — while staying grounded in the official FAQ content.



In [21]:
!pip install requests python-dotenv

In [22]:
from google.colab import userdata

API_KEY = userdata.get("OPENROUTER_API_KEY")

In [23]:
import requests
import json

In [24]:
API_URL = "https://openrouter.ai/api/v1/chat/completions"

MODEL = "openai/gpt-4o-mini"

In [25]:
import requests

class LLMClient:

    def __init__(self, api_key, model):
        self.api_key = api_key
        self.model = model

    def generate(self, prompt, system_message=None, max_tokens=512):

        headers = {
            "Authorization": f"Bearer {self.api_key}",
            "Content-Type": "application/json"
        }

        payload = {
            "model": self.model,
            "messages": []
        }

        if system_message:
            payload["messages"].append(
                {
                    "role": "system",
                    "content": system_message
                }
            )

        payload["messages"].append(
            {
                "role": "user",
                "content": prompt
            }
        )

        payload["max_tokens"] = max_tokens

        try:

            response = requests.post(
                API_URL,
                headers=headers,
                json=payload
            )

            response.raise_for_status()

            result = response.json()

            return result["choices"][0]["message"]["content"]

        except requests.exceptions.HTTPError as e:

            return f"HTTP Error: {e}\n{response.text}"

        except Exception as e:

            return f"Error: {e}"

    def generate_faq_response(self, user_question, faq_entry):

        system_prompt = """
You are SupportAI.

Rules:
1. Answer ONLY from the provided FAQ.
2. Never invent information.
3. If the FAQ does not fully answer the user's question, politely say that the information is not available in the FAQ.
4. Use a friendly and professional support-agent tone.
5. Keep the response under 150 words.
"""

        prompt = f"""
Official FAQ

FAQ Question:
{faq_entry["question"]}

Official FAQ Answer:
{faq_entry["answer"]}

User Question:
{user_question}
"""

        return self.generate(
            prompt=prompt,
            system_message=system_prompt,
            max_tokens=150
        )

In [26]:
client = LLMClient(
    api_key=API_KEY,
    model=MODEL
)

In [27]:
user_question = "I can't remember my login password"

results = search_by_keyword(faqs, user_question)

if results:

    faq = results[0]

    print("Question:")
    print(user_question)

    print("\nMatched FAQ:")
    print(faq["question"])

    answer = client.generate_faq_response(
        user_question,
        faq
    )

    print("\nSupportAI Response:")
    print(answer)

else:

    print("No FAQ found.")

Question:
I can't remember my login password

Matched FAQ:
How do I reset my password?

SupportAI Response:
I can help with that! To reset your password, please click on 'Forgot Password' on the login page. Then, enter your registered email address and follow the instructions provided for resetting your password. If you have any further questions, feel free to ask!


In [28]:
client = LLMClient(
    api_key=API_KEY,
    model=MODEL
)

In [29]:
print(hasattr(client, "generate_faq_response"))

True


In [30]:
user_question = "I can't remember my login password"

results = search_by_keyword(faqs, user_question)

if results:

    faq = results[0]

    print("Question:")
    print(user_question)

    print("\nMatched FAQ:")
    print(faq["question"])

    answer = client.generate_faq_response(
        user_question,
        faq
    )

    print("\nSupportAI Response:")
    print(answer)

else:
    print("No FAQ found.")

Question:
I can't remember my login password

Matched FAQ:
How do I reset my password?

SupportAI Response:
To reset your password, please click 'Forgot Password' on the login page. Then, enter your registered email address and follow the provided password reset instructions. This will help you regain access to your account. If you have any other questions, feel free to ask!


In [31]:
user_question = "Can I get a refund for my annual subscription?"

results = search_by_keyword(faqs, user_question)

if results:

    faq = results[0]

    print("Question:")
    print(user_question)

    print("\nMatched FAQ:")
    print(faq["question"])

    answer = client.generate_faq_response(
        user_question,
        faq
    )

    print("\nSupportAI Response:")
    print(answer)

else:
    print("No FAQ found.")

Question:
Can I get a refund for my annual subscription?

Matched FAQ:
How do I reset my password?

SupportAI Response:
I'm sorry, but the information regarding refunds for annual subscriptions is not available in the FAQ. If you need assistance with this issue, I recommend reaching out to customer support directly for further help.


In [32]:
invalid_client = LLMClient(
    api_key="INVALID_API_KEY",
    model=MODEL
)

response = invalid_client.generate("Hello")

print(response)

HTTP Error: 401 Client Error: Unauthorized for url: https://openrouter.ai/api/v1/chat/completions
{"error":{"message":"Missing Authentication header","code":401}}


# Prompt Design

The system prompt is designed to ensure the LLM produces reliable,
grounded responses.

Rules included in the prompt:

1. Answer only using the provided FAQ content.
2. Do not invent or assume information beyond the FAQ.
3. If the FAQ does not fully answer the user's question,
   politely inform the user that the information is unavailable.
4. Respond in a friendly and professional support-agent tone.
5. Keep responses concise and under 150 words.

This prompt minimizes hallucinations by restricting the model to
the official FAQ content while still producing natural,
conversational responses.

In [33]:
print(search_by_keyword(faqs, "forgot my password")[0]["question"])

print(get_faq_by_id(faqs, "faq-001")["question"])

print(get_faqs_by_category(faqs, "Account")[0]["question"])

How do I reset my password?
How do I reset my password?
How do I reset my password?


# Task 3 - Intelligent FAQ Matching

## Step 1: Install Required Library

In this task, we improve the FAQ retrieval system by using **TF-IDF (Term Frequency–Inverse Document Frequency)** and **Cosine Similarity**.

Unlike keyword search, TF-IDF measures the importance of words in each FAQ and allows the system to retrieve relevant FAQs even when users phrase their questions differently.

For this implementation, we use the **scikit-learn** library, which provides built-in implementations of both TF-IDF vectorization and cosine similarity.

In [34]:
!pip install scikit-learn

## Step 2: Import Required Libraries

This task uses two components from scikit-learn:

- **TfidfVectorizer**: Converts FAQ text into numerical TF-IDF vectors.
- **cosine_similarity**: Measures how similar a user's question is to each FAQ.

These libraries allow us to build a simple semantic retrieval system without using a large language model.

In [35]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

## Step 3: Build the FAQMatcher Class

The `FAQMatcher` class is responsible for finding the most relevant FAQ using TF-IDF.

When an object of this class is created:

1. The FAQ data is stored.
2. Each FAQ's **question** and **keywords** are combined into a single text document.
3. A TF-IDF vectorizer learns the vocabulary from all FAQs.
4. Every FAQ is converted into a TF-IDF vector.

These vectors form the searchable index that will be used to compare future user queries.

In [36]:
class FAQMatcher:

    def __init__(self, faqs):

        self.faqs = faqs

        self.corpus = [
            f"{faq['question']} {' '.join(faq['keywords'])}"
            for faq in faqs
        ]

        self.vectorizer = TfidfVectorizer()

        self.faq_vectors = self.vectorizer.fit_transform(self.corpus)

## Step 4: Create the FAQMatcher Object

After defining the class, we create an instance of `FAQMatcher`.

During initialization, the constructor automatically builds the TF-IDF index for all FAQs.

This index is stored in memory and will be reused whenever a user asks a question, making the search process efficient.

In [37]:
matcher = FAQMatcher(faqs)

## Step 5: Verify the TF-IDF Index

Before implementing the search methods, we verify that the TF-IDF index has been created successfully.

- The **corpus** shows the processed FAQ documents.
- The **matrix shape** shows:
  - Number of rows = Number of FAQs
  - Number of columns = Number of unique terms in the vocabulary

In [38]:
print(matcher.corpus)

['How do I reset my password? password forgot reset login account', 'What is your refund policy? refund billing payment money subscription', 'How can I track my order? track delivery order shipping', 'Why is the app crashing? crash technical bug app error', 'How do I change my email address? email change account update']


In [39]:
print(matcher.faq_vectors.shape)

(5, 34)


# Step 6–9: Implement the FAQMatcher Class

In this step, we implement the complete **FAQMatcher** class, which performs intelligent FAQ retrieval using **TF-IDF vectorization** and **Cosine Similarity**.

Unlike keyword-based search, this approach compares the user's question with every FAQ based on the importance of words and their similarity in vector space.

The class provides the following methods:

### `__init__(faqs)`
- Stores the FAQ knowledge base.
- Combines each FAQ's **question** and **keywords** into a single text document.
- Builds a TF-IDF vocabulary using `TfidfVectorizer`.
- Converts every FAQ into a TF-IDF vector to create a searchable index.

### `match(query, top_k=3)`
- Converts the user's query into a TF-IDF vector.
- Computes cosine similarity between the query and all FAQs.
- Ranks the FAQs based on similarity scores.
- Returns the top `k` matching FAQs with confidence scores.

### `best_match(query, threshold=0.15)`
- Returns only the highest-ranked FAQ.
- Applies a confidence threshold to avoid returning unrelated FAQs.
- Returns `None` if no FAQ satisfies the threshold.

### `explain_match(query)`
- Displays the top matching FAQs along with their similarity scores.
- Helps understand and evaluate how the retrieval system ranks FAQs.

After implementing the class, we test:
- The top matching FAQs using `match()`
- The best FAQ using `best_match()`
- The ranking explanation using `explain_match()`

This completes the intelligent FAQ retrieval component and prepares the system for implementing **Hybrid Search** in the next step.

In [40]:
class FAQMatcher:

    def __init__(self, faqs):

        self.faqs = faqs

        self.corpus = [
            f"{faq['question']} {' '.join(faq['keywords'])}"
            for faq in faqs
        ]

        self.vectorizer = TfidfVectorizer()

        self.faq_vectors = self.vectorizer.fit_transform(self.corpus)

    def match(self, query, top_k=3):

        query_vector = self.vectorizer.transform([query])

        similarity_scores = cosine_similarity(
            query_vector,
            self.faq_vectors
        )[0]

        results = []

        for faq, score in zip(self.faqs, similarity_scores):

            results.append(
                (
                    faq,
                    round(float(score), 4)
                )
            )

        results.sort(
            key=lambda x: x[1],
            reverse=True
        )

        return results[:top_k]

    def best_match(self, query, threshold=0.15):

        matches = self.match(query, top_k=1)

        if not matches:
            return None

        faq, score = matches[0]

        if score >= threshold:
            return faq, score

        return None

    def explain_match(self, query):

        matches = self.match(query)

        explanation = []

        for index, (faq, score) in enumerate(matches, start=1):

            explanation.append(
                f"{index}. [{score}] {faq['question']}"
            )

        return "\n".join(explanation)

In [41]:
matcher = FAQMatcher(faqs)

In [42]:
result = matcher.best_match("I forgot my login credentials")

if result:

    faq, score = result

    print("Best Match:")
    print(faq["question"])
    print("Confidence Score:", score)

else:

    print("No suitable FAQ found.")

Best Match:
How do I reset my password?
Confidence Score: 0.448


# Step 10: Implement Hybrid Search

Although TF-IDF retrieval performs better than keyword search, both approaches have their own strengths.

- **Keyword Search** performs well when the user's query contains the exact words stored in the FAQ.
- **TF-IDF Matching** performs better when the wording is different but still shares important terms.

To improve retrieval accuracy, we combine both approaches into a **Hybrid Search** system.

The hybrid search algorithm:

1. Performs keyword search using the function developed in Task 1.
2. Performs TF-IDF matching using the `FAQMatcher` class.
3. Assigns a base score of **0.5** to keyword matches.
4. Uses cosine similarity scores from TF-IDF matching.
5. Merges duplicate FAQs and keeps the highest score.
6. Returns the top matching FAQs sorted by confidence score.

This approach provides more robust FAQ retrieval than using either method alone.

In [43]:
def hybrid_search(faqs, query, top_k=3):

    matcher = FAQMatcher(faqs)

    keyword_results = search_by_keyword(faqs, query)

    tfidf_results = matcher.match(query, top_k=len(faqs))

    combined_results = {}

    for faq in keyword_results:

        combined_results[faq["id"]] = (faq, 0.5)

    for faq, score in tfidf_results:

        if faq["id"] in combined_results:

            if score > combined_results[faq["id"]][1]:

                combined_results[faq["id"]] = (faq, score)

        else:

            combined_results[faq["id"]] = (faq, score)

    final_results = list(combined_results.values())

    final_results.sort(
        key=lambda x: x[1],
        reverse=True
    )

    return final_results[:top_k]

# Step 11: Test Hybrid Search

In this step, we verify that the hybrid search function combines the strengths of keyword search and TF-IDF retrieval.

The function returns the top matching FAQs along with their confidence scores after merging duplicate results and ranking them by relevance.

In [44]:
results = hybrid_search(
    faqs,
    "I forgot my login credentials"
)

for faq, score in results:

    print(f"[{score}]")
    print(faq["question"])
    print()

[0.5]
How do I reset my password?

[0.5]
How can I track my order?

[0.5]
How do I change my email address?



# Step 12: Compare Retrieval Methods

To evaluate the effectiveness of the retrieval system, we compare three different search approaches:

1. **Keyword Search**
2. **TF-IDF Matching**
3. **Hybrid Search**

The comparison demonstrates how TF-IDF and Hybrid Search improve retrieval for paraphrased user questions that keyword search may fail to identify.

In [45]:
queries = [
    "I forgot my login credentials",
    "Can I get my money back?",
    "package delivery time"
]

for query in queries:

    print("=" * 70)
    print("Query:", query)

    print("\nKeyword Search")

    keyword_results = search_by_keyword(faqs, query)

    if keyword_results:

        for faq in keyword_results:

            print("-", faq["question"])

    else:

        print("No results")

    print("\nTF-IDF Matching")

    tfidf_results = matcher.match(query)

    for faq, score in tfidf_results:

        print(f"[{score}] {faq['question']}")

    print("\nHybrid Search")

    hybrid_results = hybrid_search(faqs, query)

    for faq, score in hybrid_results:

        print(f"[{score}] {faq['question']}")

    best = matcher.best_match(query)

    print("\nBest Match")

    if best:

        faq, score = best

        print(f"{faq['question']} ({score})")

    else:

        print("No suitable FAQ found")

    print()

Query: I forgot my login credentials

Keyword Search
- How do I reset my password?
- How can I track my order?
- How do I change my email address?
- What is your refund policy?
- Why is the app crashing?

TF-IDF Matching
[0.448] How do I reset my password?
[0.0831] How can I track my order?
[0.0821] How do I change my email address?

Hybrid Search
[0.5] How do I reset my password?
[0.5] How can I track my order?
[0.5] How do I change my email address?

Best Match
How do I reset my password? (0.448)

Query: Can I get my money back?

Keyword Search
- How can I track my order?
- How do I reset my password?
- What is your refund policy?
- How do I change my email address?
- Why is the app crashing?

TF-IDF Matching
[0.2684] How can I track my order?
[0.1872] What is your refund policy?
[0.0821] How do I reset my password?

Hybrid Search
[0.5] How can I track my order?
[0.5] How do I reset my password?
[0.5] What is your refund policy?

Best Match
How can I track my order? (0.2684)

Query: 

# Task 3 Summary

In this task, we enhanced the FAQ retrieval system by introducing TF-IDF vectorization and cosine similarity.

The implementation included:

- Building a TF-IDF index for all FAQs.
- Ranking FAQs based on cosine similarity.
- Selecting the most relevant FAQ using a confidence threshold.
- Providing an explanation of the top matching FAQs.
- Combining keyword search and TF-IDF retrieval into a hybrid search system.
- Comparing the performance of keyword search, TF-IDF matching, and hybrid search using multiple user queries.

The hybrid retrieval system developed in this task will be reused in Task 4, where it will be integrated with a Large Language Model (LLM) to generate natural, grounded support responses.

# Task 4 – Complete Helpdesk Agent

## Step 1: Import Required Libraries

The final task integrates all previous components into a conversational AI helpdesk agent.

To support conversation tracking and ticket generation, we import:

- **dataclass** – stores conversation turns in a structured format.
- **random** – generates mock ticket IDs during escalation.

These libraries help organize conversation history and simulate a real customer support workflow.

In [46]:
from dataclasses import dataclass
import random

## Step 2: Create the ConversationTurn Dataclass

Each interaction between the user and SupportAI is stored as a conversation record.

Every conversation turn contains:

- The speaker (`user` or `assistant`)
- The message content
- The matched FAQ ID (if available)
- The confidence score of the retrieved FAQ

Using a dataclass makes the conversation history structured, readable, and easy to maintain.

In [47]:
@dataclass
class ConversationTurn:

    role: str
    content: str
    faq_id: str = None
    confidence: float = None

## Step 3: Build the SupportAgent Class

The `SupportAgent` class is responsible for coordinating all the components developed in the previous tasks.

During initialization, the agent receives:

- The FAQ knowledge base from Task 1.
- The LLM client from Task 2.
- A confidence threshold used to determine whether a FAQ match is reliable.

The class also initializes:

- A conversation history to store user and assistant messages.
- An escalation flag to track whether the conversation has been escalated.
- A low-confidence counter to monitor consecutive unsuccessful searches.

The `handle_message()` method processes each user query by combining hybrid search, LLM response generation, and conversation tracking.

In [48]:
class SupportAgent:

    def __init__(
        self,
        faqs,
        llm_client,
        confidence_threshold=0.15
    ):

        self.faqs = faqs
        self.llm_client = llm_client
        self.confidence_threshold = confidence_threshold

        self.history = []
        self.escalated = False
        self.low_confidence_count = 0

    def handle_message(self, user_message):

        self.history.append(
            ConversationTurn(
                role="user",
                content=user_message
            )
        )

        results = hybrid_search(
            self.faqs,
            user_message,
            top_k=1
        )

        if results:

            faq, score = results[0]

            if score >= self.confidence_threshold:

                response = self.llm_client.generate_faq_response(
                    user_message,
                    faq
                )

                self.history.append(
                    ConversationTurn(
                        role="assistant",
                        content=response,
                        faq_id=faq["id"],
                        confidence=score
                    )
                )

                self.low_confidence_count = 0

                return response, faq["id"], score

        fallback = (
            "I'm sorry, but I couldn't find an answer to your question in my FAQ knowledge base.\n"
            "Would you like me to connect you with a human support agent?"
        )

        self.history.append(
            ConversationTurn(
                role="assistant",
                content=fallback,
                faq_id=None,
                confidence=0.0
            )
        )

        self.low_confidence_count += 1

        return fallback, None, 0.0

In [49]:
print(faqs)

[{'id': 'faq-001', 'category': 'Account', 'question': 'How do I reset my password?', 'answer': "Click 'Forgot Password' on the login page. Enter your registered email address and follow the password reset instructions.", 'keywords': ['password', 'forgot', 'reset', 'login', 'account']}, {'id': 'faq-002', 'category': 'Billing', 'question': 'What is your refund policy?', 'answer': 'We offer full refunds within 30 days for unused subscriptions.', 'keywords': ['refund', 'billing', 'payment', 'money', 'subscription']}, {'id': 'faq-003', 'category': 'Delivery', 'question': 'How can I track my order?', 'answer': "Open the Orders page and click 'Track Order' to view the latest status.", 'keywords': ['track', 'delivery', 'order', 'shipping']}, {'id': 'faq-004', 'category': 'Technical', 'question': 'Why is the app crashing?', 'answer': 'Update the application to the latest version and restart your device.', 'keywords': ['crash', 'technical', 'bug', 'app', 'error']}, {'id': 'faq-005', 'category': 

## Step 4: Create the SupportAgent Object

After defining the `SupportAgent` class, we create an instance of the agent.

This object will manage all user interactions by combining:

- Hybrid FAQ retrieval
- LLM-based response generation
- Conversation history tracking
- Confidence monitoring
- Escalation handling

The same agent instance will be reused throughout the remaining steps of the project.

In [50]:
agent = SupportAgent(
    faqs,
    client
)

## Step 5: Test the handle_message() Method

In this step, we verify that the SupportAgent correctly processes a user's message.

The method should:

1. Store the user's message in the conversation history.
2. Retrieve the best FAQ using the hybrid search function.
3. Generate a natural response using the LLM if a confident match is found.
4. Store the assistant's response together with the matched FAQ ID and confidence score.
5. Return the generated response for display.

In [59]:
response, faq_id, confidence = agent.handle_message(
    "I forgot my login credentials"
)

print("SupportAI Response:\n")
print(response)

print("\nMatched FAQ:", faq_id)
print("Confidence Score:", confidence)

SupportAI Response:

I'm sorry, but the information regarding forgotten login credentials is not available in the FAQ. However, if you've forgotten your password, you can click 'Forgot Password' on the login page, enter your registered email address, and follow the instructions to reset your password. If you need further assistance, please let me know!

Matched FAQ: faq-001
Confidence Score: 0.5


## Step 6: Test the Fallback Response

To verify the agent's behavior when no suitable FAQ exists, we ask a question that is outside the FAQ knowledge base.

If no FAQ satisfies the confidence threshold, the agent should:

- Return a fallback response.
- Offer to connect the user with a human support agent.
- Record the interaction with a confidence score of `0.0`.

In [ ]:
response, faq_id, confidence = agent.handle_message(
    "What are your office hours in Tokyo?"
)

print("SupportAI Response:\n")
print(response)

print("\nMatched FAQ:", faq_id)
print("Confidence Score:", confidence)

## Step 7: Implement the `escalate()` Method

The `escalate()` method allows the SupportAgent to transfer unresolved conversations to a human support representative.

When this method is called, the agent:

- Marks the current session as escalated.
- Generates a unique mock support ticket ID.
- Returns a confirmation message with the ticket ID and the estimated response time.

This simulates the escalation process commonly used in real-world customer support systems.

In [60]:
class SupportAgent:

    def __init__(self, faqs, llm_client, confidence_threshold=0.15):

        self.faqs = faqs
        self.llm_client = llm_client
        self.confidence_threshold = confidence_threshold

        self.history = []
        self.escalated = False
        self.low_confidence_count = 0

    def handle_message(self, user_message):

        self.history.append(
            ConversationTurn(
                role="user",
                content=user_message
            )
        )

        results = hybrid_search(
            self.faqs,
            user_message,
            top_k=1
        )

        if results:

            faq, score = results[0]

            if score >= self.confidence_threshold:

                response = self.llm_client.generate_faq_response(
                    user_message,
                    faq
                )

                self.history.append(
                    ConversationTurn(
                        role="assistant",
                        content=response,
                        faq_id=faq["id"],
                        confidence=score
                    )
                )

                self.low_confidence_count = 0

                return response, faq["id"], score

        fallback = (
            "I'm sorry, but I couldn't find an answer to your question in my FAQ knowledge base.\n"
            "Would you like me to connect you with a human support agent?"
        )

        self.history.append(
            ConversationTurn(
                role="assistant",
                content=fallback,
                faq_id=None,
                confidence=0.0
            )
        )

        self.low_confidence_count += 1

        return fallback, None, 0.0

    def escalate(self, reason="User requested human support"):

        self.escalated = True

        ticket_id = f"TICKET-{random.randint(10000, 99999)}"

        return (
            f"Your request has been escalated to our support team.\n"
            f"Reason: {reason}\n"
            f"Ticket ID: {ticket_id}\n"
            f"Estimated response time: Within 4 business hours."
        )

## Step 8: Test the `escalate()` Method

This step verifies that the escalation functionality works correctly.

The test confirms that:

- The conversation is marked as escalated.
- A unique mock ticket ID is generated.
- A confirmation message is returned to the user with an estimated response time.

In [61]:
agent = SupportAgent(
    faqs,
    client
)

print(agent.escalate())

Your request has been escalated to our support team.
Reason: User requested human support
Ticket ID: TICKET-43969
Estimated response time: Within 4 business hours.


## Step 9: Implement the `get_conversation_summary()` Method

The `get_conversation_summary()` method provides a structured overview of the conversation between the user and the SupportAgent.

For every conversation turn, the summary includes:

- The speaker (User or Assistant)
- The message content
- The matched FAQ ID (if available)
- The confidence score of the retrieved FAQ

This summary is useful for reviewing conversations and can help human support agents quickly understand the interaction history after an escalation.

In [62]:
class SupportAgent:

    def __init__(self, faqs, llm_client, confidence_threshold=0.15):

        self.faqs = faqs
        self.llm_client = llm_client
        self.confidence_threshold = confidence_threshold

        self.history = []
        self.escalated = False
        self.low_confidence_count = 0

    def handle_message(self, user_message):

        self.history.append(
            ConversationTurn(
                role="user",
                content=user_message
            )
        )

        results = hybrid_search(
            self.faqs,
            user_message,
            top_k=1
        )

        if results:

            faq, score = results[0]

            if score >= self.confidence_threshold:

                response = self.llm_client.generate_faq_response(
                    user_message,
                    faq
                )

                self.history.append(
                    ConversationTurn(
                        role="assistant",
                        content=response,
                        faq_id=faq["id"],
                        confidence=score
                    )
                )

                self.low_confidence_count = 0

                return response, faq["id"], score

        fallback = (
            "I'm sorry, but I couldn't find an answer to your question in my FAQ knowledge base.\n"
            "Would you like me to connect you with a human support agent?"
        )

        self.history.append(
            ConversationTurn(
                role="assistant",
                content=fallback,
                faq_id=None,
                confidence=0.0
            )
        )

        self.low_confidence_count += 1

        return fallback, None, 0.0

    def escalate(self, reason="User requested human support"):

        self.escalated = True

        ticket_id = f"TICKET-{random.randint(10000,99999)}"

        return (
            f"Your request has been escalated to our support team.\n"
            f"Reason: {reason}\n"
            f"Ticket ID: {ticket_id}\n"
            f"Estimated response time: Within 4 business hours."
        )

    def get_conversation_summary(self):

        summary = []

        for turn in self.history:

            summary.append(f"Role: {turn.role}")
            summary.append(f"Message: {turn.content}")

            if turn.faq_id:

                summary.append(f"FAQ ID: {turn.faq_id}")
                summary.append(f"Confidence: {turn.confidence}")

            summary.append("-" * 40)

        return "\n".join(summary)

## Step 10: Test the Conversation Summary

In this step, we simulate a short conversation and verify that the SupportAgent correctly records each interaction.

The generated summary should display:

- User messages
- Assistant responses
- Matched FAQ IDs
- Confidence scores

This confirms that the conversation history is being stored and formatted correctly.

In [63]:
agent = SupportAgent(
    faqs,
    client
)

agent.handle_message("I forgot my login credentials")

agent.handle_message("Can I get my money back?")

print(agent.get_conversation_summary())

Role: user
Message: I forgot my login credentials
----------------------------------------
Role: assistant
Message: I'm sorry to hear that you're having trouble with your login credentials. Unfortunately, the information regarding retrieving your username or other login details is not available in the FAQ. However, you may want to try the 'Forgot Password' option to reset your password if you remember your email address associated with the account. If you need further assistance, please contact support directly.
FAQ ID: faq-001
Confidence: 0.5
----------------------------------------
Role: user
Message: Can I get my money back?
----------------------------------------
Role: assistant
Message: I'm sorry, but the information regarding refunds is not available in the FAQ. If you have further questions about returns or refunds, please feel free to reach out to customer support for assistance.
FAQ ID: faq-003
Confidence: 0.5
----------------------------------------


## Step 11: Implement the `reset()` Method

The `reset()` method starts a new conversation by clearing the current session.

When called, it:

- Clears the conversation history.
- Resets the escalation status.
- Resets the low-confidence counter.

This allows the SupportAgent to begin a fresh conversation without retaining information from previous interactions.

In [64]:
class SupportAgent:

    def __init__(self, faqs, llm_client, confidence_threshold=0.15):

        self.faqs = faqs
        self.llm_client = llm_client
        self.confidence_threshold = confidence_threshold

        self.history = []
        self.escalated = False
        self.low_confidence_count = 0

    def handle_message(self, user_message):

        self.history.append(
            ConversationTurn(
                role="user",
                content=user_message
            )
        )

        results = hybrid_search(
            self.faqs,
            user_message,
            top_k=1
        )

        if results:

            faq, score = results[0]

            if score >= self.confidence_threshold:

                response = self.llm_client.generate_faq_response(
                    user_message,
                    faq
                )

                self.history.append(
                    ConversationTurn(
                        role="assistant",
                        content=response,
                        faq_id=faq["id"],
                        confidence=score
                    )
                )

                self.low_confidence_count = 0

                return response, faq["id"], score

        fallback = (
            "I'm sorry, but I couldn't find an answer in my FAQ knowledge base.\n"
            "Would you like me to connect you with a human support agent?"
        )

        self.history.append(
            ConversationTurn(
                role="assistant",
                content=fallback,
                faq_id=None,
                confidence=0.0
            )
        )

        self.low_confidence_count += 1

        return fallback, None, 0.0

    def escalate(self, reason="User requested human support"):

        self.escalated = True

        ticket_id = f"TICKET-{random.randint(10000,99999)}"

        return (
            f"Your request has been escalated to our support team.\n"
            f"Reason: {reason}\n"
            f"Ticket ID: {ticket_id}\n"
            f"Estimated response time: Within 4 business hours."
        )

    def get_conversation_summary(self):

        summary = []

        for turn in self.history:

            summary.append(f"Role: {turn.role}")
            summary.append(f"Message: {turn.content}")

            if turn.faq_id:

                summary.append(f"FAQ ID: {turn.faq_id}")
                summary.append(f"Confidence: {turn.confidence}")

            summary.append("-" * 40)

        return "\n".join(summary)

    def reset(self):

        self.history = []

        self.escalated = False

        self.low_confidence_count = 0

## Step 12: Test the `reset()` Method

This step verifies that the SupportAgent can start a new conversation.

After calling `reset()`:

- The conversation history should be empty.
- Escalation status should be reset.
- The low-confidence counter should return to zero.

In [65]:
agent = SupportAgent(
    faqs,
    client
)

agent.handle_message("I forgot my login credentials")

print("Conversation Before Reset:\n")
print(agent.get_conversation_summary())

agent.reset()

print("\nConversation After Reset:\n")
print(agent.get_conversation_summary())

Conversation Before Reset:

Role: user
Message: I forgot my login credentials
----------------------------------------
Role: assistant
Message: I'm sorry, but the FAQ does not provide information on how to recover forgotten login credentials. However, I recommend checking the 'Forgot Password' option on the login page for assistance with resetting your password. If you need further help, please reach out to customer support.
FAQ ID: faq-001
Confidence: 0.5
----------------------------------------

Conversation After Reset:




## Step 13: Build the Interactive Chat Interface

The interactive chat interface allows users to communicate with the SupportAgent in a conversational manner.

The interface supports the following commands:

- **history** – Displays the conversation summary.
- **escalate** – Transfers the conversation to a human support agent.
- **reset** – Starts a new conversation.
- **quit** – Exits the application.

For every user question, the agent:

1. Searches the FAQ knowledge base using Hybrid Search.
2. Generates a natural response using the LLM.
3. Displays the matched FAQ ID and confidence score.
4. Suggests escalation after three consecutive low-confidence responses.

In [66]:
agent = SupportAgent(faqs, client)

print("=" * 50)
print("        SupportAI - Helpdesk Agent")
print("=" * 50)

while True:

    user_input = input("\nYou: ")

    if user_input.lower() == "quit":

        print("\nSupportAI:")
        print("Thank you for using SupportAI. Goodbye!")
        break

    elif user_input.lower() == "history":

        print(agent.get_conversation_summary())

    elif user_input.lower() == "reset":

        agent.reset()
        print("Conversation has been reset.")

    elif user_input.lower() == "escalate":

        print(agent.escalate())

    else:

        response, faq_id, confidence = agent.handle_message(user_input)

        print("\nSupportAI:")

        print(response)

        print(f"\nConfidence Score: {confidence}")

        print(f"Matched FAQ: {faq_id}")

        if agent.low_confidence_count >= 3:

            print("\nSupportAI:")
            print("You've had several questions outside my knowledge base.")
            print("I recommend connecting with a human support agent.")

        SupportAI - Helpdesk Agent

You: How do I reset my password?

SupportAI:
To reset your password, click 'Forgot Password' on the login page. Then, enter your registered email address and follow the password reset instructions. If you have any further questions, feel free to ask!

Confidence Score: 0.8433
Matched FAQ: faq-001

You: I forgot my login credentials

SupportAI:
I'm sorry, but the information regarding forgetting login credentials is not available in the FAQ. However, you might try using the password reset option by clicking 'Forgot Password' on the login page. If you need further assistance, please let me know!

Confidence Score: 0.5
Matched FAQ: faq-001

You: What are your office hours in Tokyo?

SupportAI:
I'm sorry, but the information regarding office hours in Tokyo is not available in the FAQ. If you have any other questions or need further assistance, feel free to ask!

Confidence Score: 0.5
Matched FAQ: faq-002

You: escalate
Your request has been escalated to 

# Task 4 Summary

In this task, we integrated all previous components into a complete conversational helpdesk system.

The final SupportAgent includes:

- Multi-turn conversation management
- Hybrid FAQ retrieval
- LLM-generated responses
- Confidence-based answer selection
- Human support escalation
- Conversation history tracking
- Session reset functionality
- Interactive command-line chat interface

This completes the SupportAI project by combining the FAQ knowledge base, intelligent retrieval system, and Large Language Model into a single end-to-end AI-powered customer support assistant.